# Step 2: 정제 (raw → staged) + apt_id_map 빌드**목적**:1. raw XML들을 정제된 Parquet로 변환 (타입 캐스팅·결측 처리·해제 거래 플래그)2. 모든 거래에 통합 단지 ID(`apt_id`) 부여3. K-apt 단지정보와 RTMS 거래 단지명을 fuzzy 매칭해 `apt_id_map.parquet` 빌드**입력**: `raw/rtms_trade/*/*/*.xml`, `staged/apt_info_snapshots/snapshot=*.parquet`**출력**:- `staged/trade/year=*/month=*/sigungu=*/part-0001.parquet`- `master/apt_id_map.parquet` (단지 통합 매핑)**핵심 함정** (설계서 3.3·6.2절):- 매매 데이터의 단지명은 표기 변형이 다양 → 정규화 후 fuzzy 매칭- `해제여부=Y`는 보존하되 `is_canceled` 플래그로 명시- 멱등성: 같은 raw를 두 번 정제해도 결과 동일

## 1. 환경 로드

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import sys
    sys.path.insert(0, '/content/drive/MyDrive/realestate_reco')
except ImportError:
    pass  # 로컬 환경

from common import *
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
print('환경 준비 완료')

## 2. raw XML → DataFrame 정제타입 캐스팅, 결측 처리, 컬럼 표준화.

In [ ]:
# 한글 컬럼 → 영문 표준 컬럼COL_RENAME_TRADE = {    '거래금액':       'deal_amount_raw',   # 콤마 포함 문자열    '건축년도':       'build_year_raw',    '년':            'year_raw',    '월':            'month_raw',    '일':            'day_raw',    '법정동':        'legal_dong_name',    '아파트':        'apt_name_raw',    '전용면적':      'area_m2_raw',    '지번':          'jibun',    '지역코드':      'sigungu_code',    '층':            'floor_raw',    '도로명':        'road_name',    '도로명건물본번호코드': 'road_main_no',    '해제여부':      'is_canceled_raw',    '해제사유발생일': 'cancel_date_raw',}def clean_trade_xml(xml_path: Path) -> pd.DataFrame:    """raw XML 파일 한 개 → 정제된 DataFrame."""    xml_text = xml_path.read_text(encoding='utf-8')    root = ET.fromstring(xml_text)    items = root.findall('.//item')    rows = [{c.tag: (c.text or '').strip() for c in item} for item in items]    if not rows:        return pd.DataFrame()    df = pd.DataFrame(rows)    df = df.rename(columns=COL_RENAME_TRADE)    # 타입 캐스팅    df['deal_amount'] = (df['deal_amount_raw'].str.replace(',', '').replace('', np.nan).astype(float))  # 만원    df['build_year'] = pd.to_numeric(df['build_year_raw'], errors='coerce').astype('Int64')    df['area_m2'] = pd.to_numeric(df['area_m2_raw'], errors='coerce')    df['floor'] = pd.to_numeric(df['floor_raw'], errors='coerce').astype('Int64')    # 거래일 (계약 연월일)    df['deal_date'] = pd.to_datetime(        df['year_raw'].astype(str).str.zfill(4) + '-' +        df['month_raw'].astype(str).str.zfill(2) + '-' +        df['day_raw'].astype(str).str.zfill(2),        errors='coerce'    )    # 해제 거래 플래그    df['is_canceled'] = df['is_canceled_raw'].fillna('').str.upper().eq('Y')    df['cancel_date'] = pd.to_datetime(df['cancel_date_raw'], errors='coerce', format='%y.%m.%d')    # 평형 버킷    df['pyeong_bucket'] = df['area_m2'].apply(        lambda x: to_pyeong_bucket(x) if pd.notna(x) else None    )    # 단지명 정규화    df['apt_name_norm'] = df['apt_name_raw'].apply(normalize_apt_name)    # 사용 컬럼만 추리기    keep = ['sigungu_code', 'legal_dong_name', 'jibun', 'road_name', 'road_main_no',            'apt_name_raw', 'apt_name_norm',            'deal_date', 'deal_amount', 'area_m2', 'pyeong_bucket', 'floor', 'build_year',            'is_canceled', 'cancel_date']    df = df[[c for c in keep if c in df.columns]]    return df# 단일 파일 테스트sample_xmls = list(DIRS['raw_trade'].rglob('*.xml'))if sample_xmls:    df_test = clean_trade_xml(sample_xmls[0])    print(f'정제 결과: {len(df_test)}건')    df_test.head()else:    print('raw_trade 디렉토리가 비어있음. Step 1a 먼저 실행.')

## 3. 법정동코드 매핑RTMS 응답엔 시군구코드(5자리)와 법정동명만 있고 법정동코드(10자리)는 없다.행정표준코드 매핑 테이블이 필요한데, 여기서는 법정동명 기반으로 추정한다.**실운영**: 행안부 ‘행정표준코드관리시스템’에서 법정동코드 전체 다운로드 후 매핑 테이블 생성.

In [ ]:
# 간소화된 매핑 — 실운영에서는 행정표준코드 전체 테이블 사용# 여기서는 시군구코드 + 법정동명만으로 임시 코드 생성def make_legal_dong_code(sigungu_code: str, legal_dong_name: str) -> str:    """임시 법정동 식별자. 실제 10자리 법정동코드 매핑 테이블이 있다면 그걸 사용."""    import hashlib    raw = f'{sigungu_code}|{(legal_dong_name or "").strip()}'    h = hashlib.sha256(raw.encode('utf-8')).hexdigest()[:5]    return f'{sigungu_code}{h}'# 테스트print(make_legal_dong_code('11680', '대치동'))print(make_legal_dong_code('11680', '역삼동'))

## 4. apt_id 부여

In [ ]:
def attach_apt_id(df: pd.DataFrame) -> pd.DataFrame:    """DataFrame의 모든 거래에 apt_id 부여."""    df = df.copy()    df['legal_dong_code'] = df.apply(        lambda r: make_legal_dong_code(r['sigungu_code'], r['legal_dong_name']),        axis=1    )    # 도로명주소 조합 (없을 수도 있음)    df['road_address'] = df.apply(        lambda r: f'{r.get("road_name", "") or ""} {r.get("road_main_no", "") or ""}'.strip(),        axis=1    )    df['apt_id'] = df.apply(        lambda r: make_apt_id(r['legal_dong_code'], r['apt_name_raw'], r['road_address']),        axis=1    )    return df# 테스트if sample_xmls:    df_test_id = attach_apt_id(df_test)    print('apt_id 부여 결과:')    print(df_test_id[['apt_name_raw', 'apt_name_norm', 'legal_dong_code', 'apt_id']].head())

## 5. staged 디렉토리에 파티션별 저장설계서 2.3절 — `year/month/sigungu` 3단 파티션.

In [ ]:
def save_staged_trade(df: pd.DataFrame, year: str, month: str, sigungu_code: str):    """한 (year, month, sigungu) 파티션 저장."""    out_dir = DIRS['staged_trade'] / f'year={year}' / f'month={month}' / f'sigungu={sigungu_code}'    out_dir.mkdir(parents=True, exist_ok=True)    out_path = out_dir / 'part-0001.parquet'    # 멱등성: 기존 파일 있으면 그냥 덮어쓰기 (raw 그대로 정제하므로 결과 동일)    df.to_parquet(out_path, index=False, compression='zstd')    return out_pathdef stage_all_raw_trade(year: str = None, month: str = None) -> pd.DataFrame:    """raw → staged 일괄 정제. (year, month) None이면 전체."""    pattern = '*.xml'    if year and month:        search_dir = DIRS['raw_trade'] / year / month    else:        search_dir = DIRS['raw_trade']    results = []    for xml_path in search_dir.rglob(pattern):        try:            # 파일명에서 시군구·연월 파싱: 11680_202604.xml            stem = xml_path.stem            sigungu_code, ym = stem.split('_')            yr, mo = ym[:4], ym[4:]            df = clean_trade_xml(xml_path)            if len(df) == 0:                results.append({'path': str(xml_path), 'rows': 0, 'status': 'EMPTY'})                continue            df = attach_apt_id(df)            out_path = save_staged_trade(df, yr, mo, sigungu_code)            results.append({                'path': str(xml_path), 'rows': len(df),                'staged': str(out_path), 'status': 'OK'            })        except Exception as e:            results.append({'path': str(xml_path), 'rows': 0,                            'status': 'ERROR', 'error': str(e)})    summary = pd.DataFrame(results)    print(f'정제 완료: {(summary.status == "OK").sum()}개 파일, {summary.rows.sum():,}건')    if (summary.status == 'ERROR').any():        print('에러:')        print(summary[summary.status == 'ERROR'])    return summary# 전체 정제 실행stage_summary = stage_all_raw_trade()

## 6. apt_id_map 빌드설계서 3.6절. 거래 데이터 + K-apt 단지정보를 fuzzy 매칭해 통합 키 테이블 생성.이 테이블이 4~7단계 모든 join의 기준.

In [ ]:
from difflib import SequenceMatcherdef name_similarity(a: str, b: str) -> float:    """단지명 유사도. 정규화된 이름으로 SequenceMatcher 비율."""    if not a or not b:        return 0.0    return SequenceMatcher(None, a, b).ratio()def build_apt_id_map(min_score: float = 0.85) -> pd.DataFrame:    """staged 거래 + 최신 단지정보 스냅샷을 합쳐 통합 매핑 빌드.    절차 (설계서 3.6절):      1. staged 거래에서 unique (apt_id, legal_dong_code, apt_name_norm, road_address) 추출      2. K-apt 단지정보를 (정규화 단지명 + 도로명) 기반으로 fuzzy 매칭      3. 점수 ≥0.85 자동 통합, 0.70~0.85 수동 검수 큐, 0.70 미만 RTMS 단독    """    # 1. 거래 데이터의 unique apt    trade_files = list(DIRS['staged_trade'].rglob('*.parquet'))    if not trade_files:        print('staged 거래 파일 없음. Step 2.5 먼저 실행.')        return pd.DataFrame()    dfs = [pd.read_parquet(p) for p in trade_files]    df_trade = pd.concat(dfs, ignore_index=True)    apt_unique = (df_trade                  .groupby(['apt_id', 'sigungu_code', 'legal_dong_code',                            'legal_dong_name', 'apt_name_norm', 'road_address'])                  .size()                  .reset_index(name='trade_count'))    print(f'거래 단지 수: {len(apt_unique):,}')    # 2. 최신 K-apt 스냅샷    apt_snaps = sorted(DIRS['staged_apt_info'].glob('snapshot=*.parquet'))    if not apt_snaps:        print('K-apt 스냅샷 없음. apt_id_map은 RTMS 단독으로 생성.')        df_map = apt_unique.copy()        df_map['kapt_code'] = None        df_map['match_score'] = 0.0        df_map['match_status'] = 'rtms_only'        return df_map    df_kapt = pd.read_parquet(apt_snaps[-1])  # 가장 최근    df_kapt['kapt_name_norm'] = df_kapt['kaptName'].apply(normalize_apt_name)    # 3. 각 거래 단지 → 같은 시군구 안에서 K-apt 후보들과 매칭    matches = []    for _, row in apt_unique.iterrows():        candidates = df_kapt[df_kapt['sigungu_code'] == row['sigungu_code']]        if len(candidates) == 0:            matches.append({**row.to_dict(), 'kapt_code': None,                            'match_score': 0.0, 'match_status': 'no_candidate'})            continue        # 정규화 단지명 유사도        candidates = candidates.copy()        candidates['score'] = candidates['kapt_name_norm'].apply(            lambda x: name_similarity(row['apt_name_norm'], x)        )        best = candidates.loc[candidates['score'].idxmax()]        if best['score'] >= min_score:            status = 'auto_matched'        elif best['score'] >= 0.70:            status = 'review_needed'        else:            status = 'rtms_only'        matches.append({            **row.to_dict(),            'kapt_code': best['kaptCode'] if status != 'rtms_only' else None,            'match_score': float(best['score']),            'match_status': status,        })    df_map = pd.DataFrame(matches)    print(f'\n매칭 결과:')    print(df_map['match_status'].value_counts())    return df_mapdf_apt_map = build_apt_id_map()df_apt_map.head()

## 7. apt_id_map 저장 + 검수 큐 출력

In [ ]:
def save_apt_id_map(df_map: pd.DataFrame):    out_path = DIRS['master'] / 'apt_id_map.parquet'    df_map.to_parquet(out_path, index=False, compression='zstd')    print(f'저장: {out_path} ({len(df_map):,}개 단지)')def export_review_queue(df_map: pd.DataFrame):    """수동 검수 필요한 매칭(0.70~0.85)을 CSV로 export — 사람이 30분 검수."""    review = df_map[df_map['match_status'] == 'review_needed'].copy()    out_path = DIRS['logs'] / f'apt_match_review_{datetime.now().strftime("%Y%m%d")}.csv'    review.to_csv(out_path, index=False, encoding='utf-8-sig')    print(f'검수 큐 저장: {out_path} ({len(review)}건)')    print('각 행에 OK/REJECT 컬럼 채워서 다시 import 하면 됨')save_apt_id_map(df_apt_map)export_review_queue(df_apt_map)

---## 다음 단계- **Step 3**: 모든 staged 데이터 + apt_id_map → master 통합 테이블 빌드  - `unified_apt_pyeong_daily`: 일별 스냅샷 상태 테이블 (설계서 5.2절)  - `trade_events`: 거래 이벤트 테이블 (설계서 5.3절)